# ML-04: Search Intelligence Data Contract

Week 3. Lane 2 (Refresh / Content Opportunity Scoring) moves from the starter CSV to the warehouse release. This notebook writes down what my slice of the warehouse is, proves three facts about it with queries, builds the first five features, and then deliberately breaks the setup with a leaked column to see what that looks like.

Setup: `pip install -r requirements.txt` (duckdb, huggingface_hub, scikit-learn), accept the gate on the `FlyRank/internship-warehouse` page, and provide a Read token as `HF_TOKEN` (environment variable or Colab secret). The loader in `work/scripts/warehouse.py` downloads each monthly file once and caches it; the token is never printed. I iterate on March 2026 as the card asks, and June 2026 stays sealed for the final test in week 6.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

def find_repo_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "work" / "scripts" / "warehouse.py").exists():
            return folder
    raise FileNotFoundError("Run this from inside a clone of the repo.")

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "work" / "scripts"))
import warehouse as W  # shared loader + Lane 2 frame builder (weeks 3-6)

con = W.connect()

## 1. Unit of analysis + time window

The contract, in five plain answers.

**What one row means.** One page (`content_hash_id`) at a decision date. For this notebook the decision date is 1 April 2026: a reviewer sits down on that morning with everything known up to 31 March. A page is in the slice if it earned at least 100 search impressions in March, was created at least 30 days before the decision date, and belongs to a client whose Search Console data covers every day of both March and April.

**Which tables.** `fact_content_daily_performance` (partitions `month=2026-03` for features and `month=2026-04` for the label only) and `dim_content` for the page's creation date. `dim_clients` isn't needed for the frame itself, because tracking coverage is checked directly on the daily rows (section 3 explains why that's the safer check).

**Which time window.** Features: 1 to 31 March 2026. Label: 1 to 30 April 2026. The windows touch but never share a day. In week 2 I planned 90-day feature windows; I've cut that to one calendar month so each decision date maps to one partition, which gives me four decision dates (March to June) for a time-aware split later instead of one.

**What I rank.** The chance that the page declines in April: label = 1 when its per-day impressions in April are more than 20% below March and that drop is at least 20 points worse than the median change across its own client's pages. Pages are ranked by that chance for a 50-page review queue.

**What I deliberately exclude.** `content_updated_date` and `last_optimized_date` from `dim_content`. They're a snapshot from the July 2026 export, so for any page edited after 1 April they describe the future, and an edit made because a page was dropping would leak the label straight into a "staleness" feature. Section 4 counts how often that happens.

## 2. Fields: feature / label / context / excluded

Every field I touch, sorted once. Anything not listed here isn't used.

In [2]:
fields = pd.DataFrame([
    ("content_hash_id, client_hash_id", "context", "join, group, client-grouped splits; never learned from"),
    ("report_date", "context", "places each daily row in the feature or label window"),
    ("gsc_impressions (March)", "feature source", "-> log_imp, days_visible, momentum"),
    ("gsc_avg_position (March)", "feature source", "-> avg_position (impression-weighted)"),
    ("gsc_clicks (March)", "feature source (later weeks)", "-> ctr_pct, log_clicks from week 4"),
    ("content_created_date", "feature source", "-> content_age_days; set at publication, does not move"),
    ("gsc_impressions (April)", "label", "the outcome; never a feature"),
    ("gsc_data_available", "context / filter", "row-level availability flag, filtered with IS TRUE"),
    ("ga4_* session columns", "excluded (for now)", "GA4 covers far fewer rows than GSC this month (section 3)"),
    ("sessions_ai, ai_*", "excluded", "too sparse to rank on; not part of this lane"),
    ("content_updated_date, last_optimized_date", "excluded", "July snapshot, can postdate the decision (section 4)"),
    ("provider_used, model_used", "excluded", "how the page was generated, not how it performs"),
    ("keyword_hash_id, url_hash_id", "context only", "dedup and grouping; pseudonyms carry no meaning"),
], columns=["field", "bucket", "why"])
fields

,field,bucket,why
0,"content_hash_id, client_hash_id",context,"join, group, client-grouped splits; never learned from"
1,report_date,context,places each daily row in the feature or label window
2,gsc_impressions (March),feature source,"-> log_imp, days_visible, momentum"
3,gsc_avg_position (March),feature source,-> avg_position (impression-weighted)
4,gsc_clicks (March),feature source (later weeks),"-> ctr_pct, log_clicks from week 4"
5,content_created_date,feature source,"-> content_age_days; set at publication, does not move"
6,gsc_impressions (April),label,the outcome; never a feature
7,gsc_data_available,context / filter,"row-level availability flag, filtered with IS TRUE"
8,ga4_* session columns,excluded (for now),GA4 covers far fewer rows than GSC this month (section 3)
9,"sessions_ai, ai_*",excluded,too sparse to rank on; not part of this lane


## 3. Verify it with queries (grain, counts, missing values, windows)

Exactly three verification queries, all on the mid-panel month (March 2026). Everything after them builds features; it doesn't re-verify.

In [3]:
con.execute(f"CREATE OR REPLACE VIEW march AS SELECT * FROM read_parquet('{W.month_path('2026-03')}')")

# Q1. Grain: one daily row per (date, client, page), and every page belongs to exactly one client.
q1 = con.sql("""
    SELECT
        (SELECT COUNT(*) FROM (
            SELECT report_date, client_hash_id, content_hash_id
            FROM march GROUP BY ALL HAVING COUNT(*) > 1))                 AS duplicate_daily_keys,
        (SELECT COUNT(*) FROM (
            SELECT content_hash_id
            FROM march GROUP BY 1 HAVING COUNT(DISTINCT client_hash_id) > 1)) AS pages_in_two_clients
""").df()
q1

,duplicate_daily_keys,pages_in_two_clients
0,0,0


In [4]:
# Q2. Row count and date span of the slice.
q2 = con.sql("""
    SELECT COUNT(*)                        AS daily_rows,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT report_date)     AS distinct_days,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS pages
    FROM march
""").df()
q2

,daily_rows,first_day,last_day,distinct_days,clients,pages
0,9841378,2026-03-01,2026-03-31,31,55,331437


In [5]:
# Q3. Availability: how many rows survive an IS TRUE filter on each source.
q3 = con.sql("""
    SELECT COUNT(*)                                                         AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)               AS gsc_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)               AS ga4_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                              AND ga4_data_available IS TRUE)               AS both_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                              AND gsc_impressions = 0)                      AS gsc_rows_with_zero_impressions
    FROM march
""").df()
q3["gsc_share"] = (q3["gsc_rows"] / q3["all_rows"]).round(3)
q3["ga4_share"] = (q3["ga4_rows"] / q3["all_rows"]).round(3)
q3.T.rename(columns={0: "value"})

,value
all_rows,9841378.000
gsc_rows,3611061.000
ga4_rows,413966.000
both_rows,364347.000
gsc_rows_with_zero_impressions,0.000
gsc_share,0.367
ga4_share,0.042


What the three queries show.

The grain holds: no duplicate (date, client, page) keys, and no page sits in two clients. March has 9,841,378 daily rows over all 31 days, from 55 clients and 331,437 pages.

Availability is the one that changed how I build the frame. Only 36.7% of rows survive `gsc_data_available IS TRUE`, and not one of them has zero impressions. Search Console rows only exist on days a page was actually shown. So for a client that's being tracked, "no row" means zero impressions, not missing data. That's why tracking coverage is checked per client (the client must have Search Console rows on every day of both months) rather than per page, and why a page with no April rows at a tracked client counts as zero instead of falling out of the join. My first draft got this wrong and quietly kept only pages that were visible every single day. GA4 survives on just 4.2% of rows, which is why session features wait.

### Five features, max

Built from March only, so the decision date is 1 April. The aggregation SQL lives in `work/scripts/warehouse.py` (`FEATURE_SQL`), because weeks 4 to 6 rebuild the same frame for other months and I want one definition, not four copies.

| feature | knowable at the decision moment because... |
|---|---|
| `log_imp` | it's the log of total March impressions, and March ended before 1 April |
| `days_visible` | it counts March days with at least one impression, all before 1 April |
| `avg_position` | impression-weighted mean of March daily positions, all before 1 April |
| `momentum` | log per-day impressions for 16 to 31 March minus 1 to 15 March; both halves are in March |
| `content_age_days` | days from the page's creation date to 1 April; the creation date is fixed at publication and can't be rewritten by a later edit |

In [6]:
frame = W.build_frame(con, feature_month="2026-03", label_month="2026-04")
assert frame["content_hash_id"].is_unique, "grain broken: a page appears twice"

print(f"pages in the slice: {len(frame):,}   clients: {frame['client_hash_id'].nunique()}")
print(f"label base rate (declined in April, client-relative): {frame['label'].mean():.1%}")
frame[["content_hash_id", "client_hash_id", *W.FIVE_FEATURES, "label"]].sample(6, random_state=3).round(2)

pages in the slice: 84,347   clients: 36
label base rate (declined in April, client-relative): 27.0%


,content_hash_id,client_hash_id,log_imp,days_visible,avg_position,momentum,content_age_days,label
59475,content_b4ac3066b0f99737,client_62f4a7e64f5e0096,8.35,31,2.16,0.41,267,0
38783,content_7659a45e7497d2b3,client_62f4a7e64f5e0096,5.35,30,42.31,0.04,267,0
35635,content_6cc92a28d459bcb4,client_73cda7b4e4f265ea,7.62,31,3.00,0.38,260,1
81316,content_f6cf81886903cb92,client_fef1a8f436438636,6.77,31,0.57,0.52,96,0
82289,content_f9b0832b33ee3bfe,client_2094c6eb080311d5,6.82,31,6.12,0.02,69,1
48943,content_950547f24b342c28,client_23a62021009f63c4,8.40,31,25.52,0.09,217,0


In [7]:
frame[W.FIVE_FEATURES].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
log_imp,84347.0,6.85,1.45,4.62,5.63,6.70,7.92,12.41
days_visible,84347.0,29.54,3.65,1.00,30.00,31.00,31.00,31.00
avg_position,84347.0,14.56,15.17,0.02,4.82,8.20,19.10,106.89
momentum,84347.0,0.08,0.82,-7.04,-0.32,0.02,0.36,7.00
content_age_days,84347.0,192.29,116.52,31.00,79.00,192.00,264.00,495.00


### The trap, on purpose

A quick score first: a gradient-boosted classifier on the five features, tested on clients it never trained on (30% of clients held out). Then I add ONE label-derived column, `change_pct` (the April-vs-March change the label is computed from), and run the exact same thing again.

In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

y = frame["label"]
train_idx, test_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(frame, y, frame["client_hash_id"])
)

def quick_score(columns):
    model = HistGradientBoostingClassifier(random_state=42)
    model.fit(frame.loc[train_idx, columns], y.loc[train_idx])
    return roc_auc_score(y.loc[test_idx], model.predict_proba(frame.loc[test_idx, columns])[:, 1])

honest = quick_score(W.FIVE_FEATURES)
leaky = quick_score(W.FIVE_FEATURES + ["change_pct"])
print(f"held-out clients: {frame.loc[test_idx, 'client_hash_id'].nunique()}, rows: {len(test_idx):,}, base rate {y.loc[test_idx].mean():.1%}")
print(f"ROC AUC, five honest features:          {honest:.2f}")
print(f"ROC AUC, + change_pct (label-derived):  {leaky:.2f}")

held-out clients: 11, rows: 39,414, base rate 27.7%
ROC AUC, five honest features:          0.66
ROC AUC, + change_pct (label-derived):  0.89


In [9]:
# Delete it. The honest number is the one that stays.
features = list(W.FIVE_FEATURES)
assert "change_pct" not in features
final = quick_score(features)
print(f"ROC AUC after removing the leaked column: {final:.2f}")

ROC AUC after removing the leaked column: 0.66


Adding the one label-derived column moves the score from 0.66 to 0.89 on the same held-out clients. It stops short of 1.0 because the label has a second part, the client's median change, which a model can't work out for clients it never saw. The jump is still the confession: one column, built from April, did most of the work.

With it deleted, 0.66 is the honest starting number on a 27.7% base rate. That's what the week 4 baseline and the week 5 model are measured against, and it's modest on purpose. Five features from one month shouldn't be expected to do much more.

## 4. Data limits

The limitation I'm naming for this slice: **I can't measure staleness at the decision date.** The only update dates in the release are a snapshot from the July export, so a page refreshed in May shows a May update date even when I'm pretending to stand on 1 April. The cell below counts how many pages in my slice have that problem.

In [10]:
dates = con.sql(f"""
    SELECT content_hash_id, content_updated_date, last_optimized_date
    FROM read_parquet('{W.file_path('dim_content.parquet')}')
""").df()
check = frame[["content_hash_id", "label"]].merge(dates, on="content_hash_id", how="left")
after = pd.to_datetime(check["content_updated_date"]) >= pd.Timestamp("2026-04-01")
print(f"pages whose 'last updated' date is after the decision date: {after.mean():.1%}")
print(f"decline rate when updated after 1 April: {check.loc[after, 'label'].mean():.1%}  vs  otherwise: {check.loc[~after, 'label'].mean():.1%}")

pages whose 'last updated' date is after the decision date: 79.6%
decline rate when updated after 1 April: 24.7%  vs  otherwise: 36.1%


79.6% of pages in the slice show a "last updated" date after 1 April, so the field mostly describes edits made after the decision. It isn't neutral either: pages updated after 1 April declined less often (24.7%) than the rest (36.1%). A staleness feature built from it would carry April-and-later information into a March decision, so the signal behind the refresh flags is off the table for this lane. Content age, from the fixed creation date, stands in for it, and the write-up will say so.

Other limits I'm carrying forward:

- The panel is unbalanced: only 36 of the 55 clients with March rows have Search Console coverage on every day of both months.
- One client contributes about a quarter of the pages, so client-grouped validation is required, not optional.
- A single March-to-April transition can't separate seasonality from page decline. The other decision months (February to June) are there for that in weeks 5 and 6.

## Self-check

- [x] Every section above is filled, markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere (pseudonymized hashes and aggregates only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`